# 06-2 — Improvement (option B): train and evaluate (Colab)

**Runs:** (all with the authors' settings, cached features, as in notebook 05)
| tag | attacked training images | extra distractor words in the loss | seeds | isolates |
|---|---|---|---|---|
| `vocab_seed{0,1,2}` | 50% everyday words | the 200 everyday words | 3 | **option B** |
| `ctrl_renderer` | authors' word rule, our renderer | none | 1 | effect of re-rendering (should ≈ reproduction) |
| `ctrl_distractors` | authors' word rule, our renderer | the 200 everyday words | 1 | effect of only adding the words to the loss |

**Primary test sets for option B** (not used to motivate it): real-world RTA-100, PAINT, Disentangling and the
synthetic Flowers, Food, Aircraft, DTD, EuroSAT, Cars. Pets and Caltech (where the idea came from) are reported
separately, with the held-out everyday words as a separate group. ImageNet-1K val is excluded from the option-B
comparison (its class names are the source of the everyday vocabulary). Clean accuracy must not drop.

**Before running:** T4 GPU. Drive `MyDrive/dp_project/cache_train/` must contain `train_imagenet100.pt`,
`train_attacks_vocab.pt` and `vocab.json` (from 06-1). Notebook 05 should have finished (its results are reused).

In [ ]:
# A1) Paths
from google.colab import drive
drive.mount("/content/drive")
import glob, os, sys, subprocess, json
DRIVE = "/content/drive/MyDrive/dp_project"; W = f"{DRIVE}/experiments"; BASE = "/content"
CT = f"{DRIVE}/cache_train"; CACHE = f"{DRIVE}/cache_test"
for f in ["train_imagenet100.pt", "train_attacks_vocab.pt", "vocab.json"]:
    assert os.path.exists(f"{CT}/{f}"), f"missing {CT}/{f}"
print("test sets:", sorted(os.listdir(CACHE)))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# A2)
# Dependencies, authors' code pinned to 50d7887, the two compatibility patches (same as every notebook)
!pip -q install ftfy regex git+https://github.com/openai/CLIP.git
import pathlib
os.chdir(BASE)
if not os.path.isdir("Defense-Prefix"):
    subprocess.run("git clone -q https://github.com/azuma164/Defense-Prefix.git", shell=True, check=True)
os.chdir(f"{BASE}/Defense-Prefix")
subprocess.run("git checkout -q 50d7887", shell=True, check=True)
p = pathlib.Path("datasets/utils/make_dataset_train.py"); s = p.read_text()
if "_textsize" not in s:
    helper = "\ndef _textsize(draw, text, font):\n    # Replacement for ImageDraw.textsize (removed in Pillow 10).\n    l, t, r, b = draw.multiline_textbbox((0, 0), text, font=font)\n    return r, b\n"
    s = s.replace("random.seed(300)", "random.seed(300)\n" + helper, 1).replace("imagedraw.textsize(text, font=font)", "_textsize(imagedraw, text, font)")
    p.write_text(s)
e = pathlib.Path("eval.py"); s = e.read_text()
e.write_text(s.replace("torch.load(args.tokens_path)", "torch.load(args.tokens_path, map_location=device, weights_only=False)"))
os.makedirs("data", exist_ok=True)
import torch, torchvision, PIL
print(subprocess.run("git rev-parse HEAD && git diff --stat", shell=True, capture_output=True, text=True).stdout)
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__, "| Pillow", PIL.__version__)

In [ ]:
%%writefile merge_train.py
"""Combine the clean features of train_imagenet100.pt with the new attacked versions from make_train_attacks.py.

Writes two training files in the format train_dp_cached.py expects (classes, labels, original, typographic):
  train_ctrl.pt   attacked version 'ctrl'  (authors' word rule, our renderer)          -> renderer control
  train_vocab.pt  attacked version 'vocab' (50% everyday words from v_train)           -> option B
and v_train.txt (the everyday words, used as extra distractor classes with --extra_words).
Images are matched by relative path (wnid/file); class order and labels come from train_imagenet100.pt.

With --check it also measures, on the training images, how often PLAIN CLIP predicts the written word
(raw attack strength) for ImageNet-100 words vs everyday words — the premise of option B.

Usage:
  python merge_train.py --base .../train_imagenet100.pt --attacks .../train_attacks_vocab.pt --out .../cache_train --check
"""
import argparse
import json
import os
import sys

import torch
import torch.nn.functional as F

sys.path.insert(0, os.getcwd())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", required=True)
    ap.add_argument("--attacks", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--check", action="store_true")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    base = torch.load(args.base, map_location="cpu", weights_only=False)
    att = torch.load(args.attacks, map_location="cpu", weights_only=False)
    base_rel = ["/".join(str(f).split("/")[-2:]) for f in base["files"]]
    pos = {r: i for i, r in enumerate(base_rel)}
    missing = [r for r in att["relpaths"] if r not in pos]
    assert not missing, f"{len(missing)} attacked images have no clean counterpart, e.g. {missing[:3]}"
    idx = torch.tensor([pos[r] for r in att["relpaths"]])
    classes = list(base["classes"])
    labels = base["labels"][idx]
    # sanity: the class name recorded at rendering time must equal the base label
    bad = sum(classes[int(l)] != t for l, t in zip(labels, att["true_names"]))
    assert bad == 0, f"{bad} label mismatches between the two files"
    for tag, key in (("ctrl", "typographic_ctrl"), ("vocab", "typographic_vocab")):
        torch.save({"classes": classes, "labels": labels, "original": base["original"][idx], "typographic": att[key],
                    "files": att["relpaths"], "meta": {**att["meta"], "variant": tag, "base": os.path.basename(args.base)}},
                   os.path.join(args.out, f"train_{tag}.pt"))
        print("saved", os.path.join(args.out, f"train_{tag}.pt"))
    with open(os.path.join(args.out, "v_train.txt"), "w") as f:
        f.write("\n".join(att["v_train"]))
    print(len(att["v_train"]), "everyday training words ->", os.path.join(args.out, "v_train.txt"))

    if args.check:
        import clip
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model, _ = clip.load("ViT-B/32", device=device)
        cand = classes + [w for w in att["v_train"] if w not in classes]
        with torch.no_grad():
            t = F.normalize(model.encode_text(clip.tokenize([f"a photo of a {c}." for c in cand]).to(device)), dim=-1)
            x = att["typographic_vocab"].to(device)
            pred = torch.cat([(x[i:i + 8192] @ t.T).argmax(-1).cpu() for i in range(0, len(x), 8192)])
        target = torch.tensor([cand.index(w) for w in att["word_vocab"]])
        everyday = torch.tensor([a != b for a, b in zip(att["word_ctrl"], att["word_vocab"])])
        hit = pred.eq(target).float()
        res = {"clip_attack_success_imagenet100_words": round(100 * hit[~everyday].mean().item(), 2),
               "clip_attack_success_everyday_words": round(100 * hit[everyday].mean().item(), 2),
               "n_imagenet100": int((~everyday).sum()), "n_everyday": int(everyday.sum())}
        print(json.dumps(res))
        json.dump(res, open(os.path.join(args.out, "attack_strength_check.json"), "w"), indent=1)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train_dp_cached.py
"""Train a Defense-Prefix (DP) vector on cached CLIP image features.

Re-implements the authors' train.py (commit 50d7887) with one change: image features
are read from `train_imagenet100.pt` instead of being recomputed every step.  In the
authors' code the image features are computed under `torch.no_grad()` from a frozen
encoder, so caching them gives the same loss and the same gradients for the DP vector.

Kept identical to the authors' train.py:
  * CLIP ViT-B/32, loaded with clip.load (fp16 on GPU), frozen
  * DP vector: 1 x 512, initialised N(0, 0.02), inserted in place of the "*" token
    with the authors' encode_text_with_learnt_tokens (utils/non_nv.py)
  * one random template (of the 81 in templates.py) per batch, shared by all classes
  * loss = seta * CE(attacked images, DP class features) + gamma * KL(p_orig || p_dp) on clean images
    seta = 1, gamma = 3, logits scaled by model.logit_scale.exp()
  * SGD lr 0.002, CosineAnnealingLR(T_max=epochs, eta_min=5e-5) stepped once per epoch
  * batch size 512, 10 epochs, shuffled, last partial batch kept

Small numerical difference (recorded as a deviation): the CE and KL terms are computed in
fp32 from the fp16 logits (the authors compute them directly on fp16 logits).

Added (all optional, defaults reproduce the paper):
  --seed                 seeds python/numpy/torch and the shuffle order (the authors set no seed)
  --dp_dtype fp16|fp32   authors' DP parameter is fp16 (model.dtype); fp32 is offered as a deviation
  --no_defense / --no_reg  ablations: identity loss only / defense loss only
  --extra_words FILE     extra attack words added as distractor classes (for the broad-vocabulary run)
  If `typographic` in the training file has shape (K, N, 512), one of the K attacked
  versions is drawn per epoch (attack renewal).

Usage (from the Defense-Prefix repo root, so `utils/` and `templates.py` import):
  python train_dp_cached.py --train_file /path/train_imagenet100.pt --out runs/dp_seed0 --seed 0
"""
import argparse
import json
import os
import random
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.getcwd())
import clip  # noqa: E402
from templates import imagenet_templates  # noqa: E402  (authors' 81 templates)
from utils.non_nv import encode_text_with_learnt_tokens  # noqa: E402  (authors' function)

device = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_train(path):
    d = torch.load(path, map_location="cpu", weights_only=False)
    classes = list(d["classes"])
    labels = d["labels"].long()
    original = d["original"]
    typographic = d["typographic"]
    if typographic.dim() == 2:
        typographic = typographic.unsqueeze(0)  # (1, N, 512): a single fixed attacked version (authors' setup)
    assert original.shape[0] == labels.shape[0] == typographic.shape[1]
    return classes, labels, original, typographic, d.get("meta", {})


def text_features(model, prompts, emb=None, star=None):
    toks = clip.tokenize(prompts).to(device)
    if emb is None:
        f = model.encode_text(toks)
    else:
        f = model.encode_text_with_learnt_tokens(toks, star, emb, is_emb=True)
    return F.normalize(f, dim=-1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--train_file", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch_size", type=int, default=512)
    ap.add_argument("--lr", type=float, default=0.002)
    ap.add_argument("--eta_min", type=float, default=0.00005)
    ap.add_argument("--gamma", type=float, default=3.0)
    ap.add_argument("--seta", type=float, default=1.0)
    ap.add_argument("--dp_dtype", choices=["fp16", "fp32"], default="fp16")
    ap.add_argument("--no_defense", action="store_true", help="identity (KL) loss only")
    ap.add_argument("--no_reg", action="store_true", help="defense (CE) loss only")
    ap.add_argument("--extra_words", default=None, help="text file, one extra distractor word per line")
    ap.add_argument("--max_steps", type=int, default=0, help="stop early (smoke test)")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    set_seed(args.seed)

    model, _ = clip.load("ViT-B/32", device=device)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    model.encode_text_with_learnt_tokens = type(model.encode_text)(encode_text_with_learnt_tokens, model)
    star = clip.tokenize(["*"]).to(device)[0][1]

    classes, labels, original, typographic, train_meta = load_train(args.train_file)
    extra = []
    if args.extra_words:
        with open(args.extra_words) as f:
            extra = [w.strip() for w in f if w.strip() and w.strip() not in classes]
    candidates = classes + extra  # extra words can only be wrong answers
    n, k_versions = labels.shape[0], typographic.shape[0]
    print(f"train images {n} | classes {len(classes)} | extra distractors {len(extra)} | attacked versions {k_versions}")

    # DP vector, exactly like the authors' main(): N(0, 0.02) with the model's dtype unless fp32 is requested
    dtype = model.dtype if args.dp_dtype == "fp16" else torch.float32
    init = torch.empty(1, 512, dtype=dtype, device=device)
    nn.init.normal_(init, std=0.02)
    emb = nn.Embedding.from_pretrained(init, freeze=False)
    opt = torch.optim.SGD(emb.parameters(), lr=args.lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, args.epochs, eta_min=args.eta_min)

    logit_scale = model.logit_scale.exp().detach()
    gen = torch.Generator().manual_seed(args.seed)
    history, step = [], 0
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for ep in range(args.epochs):
        version = int(torch.randint(k_versions, (1,), generator=gen)) if k_versions > 1 else 0
        order = torch.randperm(n, generator=gen)
        sums = {"loss": 0.0, "ce": 0.0, "kl": 0.0, "acc_attack_dp": 0.0}
        nb = 0
        for i in range(0, n, args.batch_size):
            idx = order[i:i + args.batch_size]
            x_clean = original[idx].to(device)
            x_typo = typographic[version, idx].to(device)
            y = labels[idx].to(device)

            template = random.choice(imagenet_templates)
            with torch.no_grad():
                t_orig = text_features(model, [template.format(c) for c in candidates])
            t_dp = text_features(model, [template.format("* " + c) for c in candidates], emb, star)

            logits_attack = logit_scale * x_typo @ t_dp.t()
            logits_orig = logit_scale * x_clean @ t_orig.t()
            logits_reg = logit_scale * x_clean @ t_dp.t()
            ce = F.cross_entropy(logits_attack.float(), y)
            kl = F.kl_div(F.log_softmax(logits_reg.float(), dim=-1), logits_orig.float().softmax(dim=-1),
                          reduction="batchmean")
            if args.no_defense:
                loss = kl
            elif args.no_reg:
                loss = ce
            else:
                loss = args.seta * ce + args.gamma * kl

            opt.zero_grad()
            loss.backward()
            opt.step()

            sums["loss"] += loss.item(); sums["ce"] += ce.item(); sums["kl"] += kl.item()
            sums["acc_attack_dp"] += (logits_attack.argmax(-1) == y).float().mean().item()
            nb += 1; step += 1
            if args.max_steps and step >= args.max_steps:
                break
        lr_used = sched.get_last_lr()[0]
        sched.step()
        rec = {"epoch": ep + 1, "lr": lr_used, "attack_version": version, "steps": nb,
               **{k: round(v / max(nb, 1), 5) for k, v in sums.items()}, "elapsed_s": round(time.time() - t0, 1)}
        history.append(rec)
        print(json.dumps(rec))
        if args.max_steps and step >= args.max_steps:
            break

    token = emb.weight.detach().to(model.dtype).cpu()  # shape (1, 512), same as the authors' saved token
    torch.save(token, os.path.join(args.out, "dp_token.pt"))
    meta = {
        "args": vars(args), "train_meta": train_meta, "n_train": n, "n_candidates": len(candidates),
        "train_seconds": round(time.time() - t0, 1),
        "peak_gpu_mem_mb": round(torch.cuda.max_memory_allocated() / 2**20, 1) if device == "cuda" else None,
        "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
        "torch": torch.__version__, "history": history,
    }
    with open(os.path.join(args.out, "train_log.json"), "w") as f:
        json.dump(meta, f, indent=1)
    print("saved", os.path.join(args.out, "dp_token.pt"), "| train seconds", meta["train_seconds"],
          "| peak MB", meta["peak_gpu_mem_mb"])


if __name__ == "__main__":
    main()

In [ ]:
%%writefile eval_cached.py
"""Evaluate CLIP and CLIP + Defense-Prefix on cached image features.

Text features and scoring follow the authors' eval.py exactly:
one template per dataset (templates[0]), "* {class}" with the * embedding
replaced by the DP vector, cosine similarity x 100, argmax.

Usage (from the Defense-Prefix repo root):
  python eval_cached.py --cache CACHE_DIR --token learned_token/dp_vit-b32.pt --tag authors_token
"""
import argparse
import csv
import glob
import json
import os
import sys

import clip
import torch
import torch.nn.functional as F

sys.path.insert(0, os.getcwd())
from utils.non_nv import encode_text_with_learnt_tokens  # authors' function

device = "cuda" if torch.cuda.is_available() else "cpu"

ORDER = ["imagenet", "caltech", "pets", "cars", "flowers", "food", "aircraft", "dtd", "sun", "eurosat",
         "disentangling", "paint", "rta-100"]
# Paper, Tables B, C and 3 (ViT-B/32): (CLIP, CLIP + DP)
PAPER = {
    "clean": {"imagenet": (62.02, 62.48), "caltech": (88.64, 89.28), "pets": (87.35, 87.22), "cars": (58.72, 57.47),
              "flowers": (66.32, 63.82), "food": (84.14, 83.65), "aircraft": (18.99, 19.26), "dtd": (44.57, 40.64),
              "sun": (61.74, 61.41), "eurosat": (42.98, 43.85)},
    "synthetic": {"imagenet": (39.10, 49.83), "caltech": (63.97, 79.54), "pets": (58.95, 72.88), "cars": (21.02, 28.64),
                  "flowers": (31.32, 44.12), "food": (56.27, 67.79), "aircraft": (10.83, 14.49), "dtd": (25.53, 31.6),
                  "sun": (34.02, 43.50), "eurosat": (4.86, 9.65)},
    "real": {"disentangling": (43.27, 71.93), "paint": (50.00, 63.64), "rta-100": (47.20, 58.00)},
}


def load_model():
    model, _ = clip.load("ViT-B/32", device=device)
    model.encode_text_with_learnt_tokens = type(model.encode_text)(encode_text_with_learnt_tokens, model)
    model.eval()
    return model


@torch.no_grad()
def class_text_features(model, classes, template, token=None):
    if token is None:
        toks = torch.cat([clip.tokenize(template.format(c)) for c in classes]).to(device)
        f = model.encode_text(toks)
        return f / f.norm(dim=-1, keepdim=True)
    toks = torch.cat([clip.tokenize(template.format(f"* {c}")) for c in classes]).to(device)
    star = clip.tokenize(["*"]).to(device)[0][1]
    f = model.encode_text_with_learnt_tokens(toks, star, token.unsqueeze(0), is_emb=False)
    return F.normalize(f, dim=-1)


@torch.no_grad()
def predictions(img_feats, text_feats, bs=8192):
    out = []
    for i in range(0, len(img_feats), bs):
        sim = 100.0 * (img_feats[i:i + bs].to(device) @ text_feats.T)
        out.append(sim.argmax(-1).cpu())
    return torch.cat(out)


def evaluate(model, cache_dir, token):
    """Returns rows: dataset, condition, n, acc_clip, acc_dp, plus per-image correctness for later stats."""
    rows, per_image = [], {}
    files = {os.path.basename(p)[:-3]: p for p in glob.glob(os.path.join(cache_dir, "*.pt"))}
    for name in [d for d in ORDER if d in files]:
        c = torch.load(files[name], map_location="cpu", weights_only=False)
        t_clip = class_text_features(model, c["classes"], c["template"])
        t_dp = class_text_features(model, c["classes"], c["template"], token)
        conds = [("real", c["clean"])] if c["real_world"] else [("clean", c["clean"]), ("synthetic", c["attack"])]
        for cond, feats in conds:
            y = c["labels"]
            ok_clip = predictions(feats, t_clip).eq(y)
            ok_dp = predictions(feats, t_dp).eq(y)
            per_image[f"{name}/{cond}"] = {"clip": ok_clip, "dp": ok_dp}
            rows.append({"dataset": name, "condition": cond, "n": len(y),
                         "acc_clip": round(100 * ok_clip.float().mean().item(), 2),
                         "acc_dp": round(100 * ok_dp.float().mean().item(), 2)})
    return rows, per_image


def with_paper(rows):
    for r in rows:
        p = PAPER.get(r["condition"], {}).get(r["dataset"])
        r["paper_clip"], r["paper_dp"] = (p if p else (None, None))
    return rows


def print_table(rows):
    print(f"{'dataset':<14}{'cond':<10}{'n':>7} | {'paper CLIP':>10}{'ours CLIP':>10} | {'paper DP':>9}{'ours DP':>9}")
    for r in rows:
        pc = "" if r["paper_clip"] is None else f"{r['paper_clip']:.2f}"
        pd = "" if r["paper_dp"] is None else f"{r['paper_dp']:.2f}"
        print(f"{r['dataset']:<14}{r['condition']:<10}{r['n']:>7} | {pc:>10}{r['acc_clip']:>10.2f} | {pd:>9}{r['acc_dp']:>9.2f}")
    for cond in ["clean", "synthetic", "real"]:
        sub = [r for r in rows if r["condition"] == cond]
        if sub:
            mc = sum(r["acc_clip"] for r in sub) / len(sub)
            md = sum(r["acc_dp"] for r in sub) / len(sub)
            print(f"  mean {cond:<9} over {len(sub)} sets: CLIP {mc:.2f} | DP {md:.2f}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", required=True)
    ap.add_argument("--token", default="learned_token/dp_vit-b32.pt")
    ap.add_argument("--tag", default="authors_token")
    ap.add_argument("--out", default=None, help="results folder (default: <cache>/../results)")
    args = ap.parse_args()

    token = torch.load(args.token, map_location=device, weights_only=False)
    token = token.detach()
    if token.dim() == 1:          # authors save shape (1, 512); eval.py passes token.unsqueeze(0)
        token = token.unsqueeze(0)
    model = load_model()
    token = token.to(device).type(model.dtype)

    rows, per_image = evaluate(model, args.cache, token)
    rows = with_paper(rows)
    print_table(rows)

    out = args.out or os.path.join(os.path.dirname(os.path.abspath(args.cache)), "results")
    os.makedirs(out, exist_ok=True)
    with open(os.path.join(out, f"{args.tag}.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    torch.save(per_image, os.path.join(out, f"{args.tag}_per_image.pt"))
    print("saved", os.path.join(out, f"{args.tag}.csv"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile eval_seen_unseen.py
"""Evaluate CLIP and one or more DP tokens on the seen/unseen test sets from make_seen_unseen.py.

Scoring is the authors' eval.py rule (dataset template, "* {class}" with * -> DP vector,
cosine x 100, argmax) over the shared candidate list (dataset classes + seen + unseen words).

Reported per dataset and token:
  acc_clean, acc_seen, acc_unseen         top-1 accuracy (%)
  hit_seen, hit_unseen                    % of images predicted as the written word (attack success)
  gap = acc_seen - acc_unseen             paired over the same images, 95% bootstrap CI
  For DP tokens additionally:
  did = (DP - CLIP)_seen - (DP - CLIP)_unseen (difference-in-differences, absolute points)
  rec_seen / rec_unseen: recovery = % of the images on which CLIP predicted the written word that the
      DP model classifies correctly (conditions on images the attack actually fooled), and the paired
      difference rec_seen - rec_unseen with a 95% bootstrap CI over images fooled in both conditions.
Per-image predictions are saved (preds.pt) for the word-level analysis in analyze_words.py.

Usage (from the Defense-Prefix repo root):
  python eval_seen_unseen.py --data cache_su/su_pets.pt cache_su/su_caltech.pt \
      --tokens authors=learned_token/dp_vit-b32.pt seed0=runs/dp_seed0/dp_token.pt --out results/seen_unseen
"""
import argparse
import csv
import json
import os
import sys

import torch
import torch.nn.functional as F

sys.path.insert(0, os.getcwd())
import clip  # noqa: E402
from utils.non_nv import encode_text_with_learnt_tokens  # noqa: E402

device = "cuda" if torch.cuda.is_available() else "cpu"


@torch.no_grad()
def text_feats(model, names, template, token=None):
    if token is None:
        f = model.encode_text(torch.cat([clip.tokenize(template.format(c)) for c in names]).to(device))
        return F.normalize(f, dim=-1)
    toks = torch.cat([clip.tokenize(template.format(f"* {c}")) for c in names]).to(device)
    star = clip.tokenize(["*"]).to(device)[0][1]
    return F.normalize(model.encode_text_with_learnt_tokens(toks, star, token.unsqueeze(0), is_emb=False), dim=-1)


@torch.no_grad()
def predict(x, t):
    return torch.cat([(100.0 * x[i:i + 8192].to(device) @ t.T).argmax(-1).cpu() for i in range(0, len(x), 8192)])


def boot_ci(diff, n_boot=2000, seed=0):
    """95% percentile bootstrap CI of the mean of a per-image difference vector (resampling images)."""
    g = torch.Generator().manual_seed(seed)
    idx = torch.randint(len(diff), (n_boot, len(diff)), generator=g)
    means = diff[idx].mean(1)
    return round(100 * means.quantile(0.025).item(), 2), round(100 * means.quantile(0.975).item(), 2)


def load_token(path, dtype):
    t = torch.load(path, map_location=device, weights_only=False).detach()
    if t.dim() == 1:
        t = t.unsqueeze(0)
    return t.to(device).type(dtype)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", nargs="+", required=True)
    ap.add_argument("--tokens", nargs="+", required=True, help="name=path pairs")
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    model, _ = clip.load("ViT-B/32", device=device)
    model.eval()
    model.encode_text_with_learnt_tokens = type(model.encode_text)(encode_text_with_learnt_tokens, model)
    tokens = {k: load_token(v, model.dtype) for k, v in (s.split("=", 1) for s in args.tokens)}

    rows = []
    for path in args.data:
        d = torch.load(path, map_location="cpu", weights_only=False)
        cand, y = d["candidates"], d["labels"]
        w_seen = torch.tensor([cand.index(w) for w in d["word_seen"]])
        w_unseen = torch.tensor([cand.index(w) for w in d["word_unseen"]])
        correct, preds = {}, {}
        for name, tok in [("clip", None)] + list(tokens.items()):
            t = text_feats(model, cand, d["template"], tok)
            p = {k: predict(d[k], t) for k in ("clean", "seen", "unseen")}
            ok = {k: p[k].eq(y).float() for k in p}
            correct[name] = ok
            preds[name] = p
            gap = ok["seen"] - ok["unseen"]
            r = {"dataset": d["dataset"], "model": name, "n": len(y),
                 "acc_clean": round(100 * ok["clean"].mean().item(), 2),
                 "acc_seen": round(100 * ok["seen"].mean().item(), 2),
                 "acc_unseen": round(100 * ok["unseen"].mean().item(), 2),
                 "hit_seen": round(100 * p["seen"].eq(w_seen).float().mean().item(), 2),
                 "hit_unseen": round(100 * p["unseen"].eq(w_unseen).float().mean().item(), 2),
                 "gap_seen_minus_unseen": round(100 * gap.mean().item(), 2)}
            r["gap_ci95"] = boot_ci(gap)
            if name != "clip":
                did = (ok["seen"] - correct["clip"]["seen"]) - (ok["unseen"] - correct["clip"]["unseen"])
                r["did"] = round(100 * did.mean().item(), 2)
                r["did_ci95"] = boot_ci(did)
                fooled_s = preds["clip"]["seen"].eq(w_seen)
                fooled_u = preds["clip"]["unseen"].eq(w_unseen)
                r["rec_seen"] = round(100 * ok["seen"][fooled_s].mean().item(), 2)
                r["rec_unseen"] = round(100 * ok["unseen"][fooled_u].mean().item(), 2)
                both = fooled_s & fooled_u
                r["n_fooled_both"] = int(both.sum())
                if both.sum() > 0:
                    dr = ok["seen"][both] - ok["unseen"][both]
                    r["rec_diff_both"] = round(100 * dr.mean().item(), 2)
                    r["rec_diff_both_ci95"] = boot_ci(dr)
            rows.append(r)
            print(json.dumps(r))
        torch.save({"preds": preds, "labels": y, "w_seen": w_seen, "w_unseen": w_unseen,
                    "word_seen": d["word_seen"], "word_unseen": d["word_unseen"]},
                   os.path.join(args.out, f"preds_{d['dataset']}.pt"))

    keys = list(dict.fromkeys(k for r in rows for k in r))
    with open(os.path.join(args.out, "seen_unseen.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)
    print("saved", os.path.join(args.out, "seen_unseen.csv"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile analyze_words.py
"""Word-level control for raw attack strength in the seen/unseen experiment.

Words differ in how strongly they fool plain CLIP (length-matching does not remove this).
For every attack word we compute, over the images it was written on:
  clip_hit  - % of images plain CLIP predicts as the written word (raw attack strength)
  dp_hit    - % of images the DP model still predicts as the written word
  dp_acc    - DP accuracy on those images
Then a weighted least-squares fit over words (weights = number of images per word):
  dp_hit = a + b * clip_hit + c * seen
c answers: at EQUAL raw attack strength, do seen words get through DP less (c < 0) or more?
A 95% CI for c comes from a bootstrap over words. A scatter plot is saved per dataset.

Usage:
  python analyze_words.py --preds results/seen_unseen_authors/preds_pets.pt [...] --model authors --out results/words
"""
import argparse
import csv
import json
import os

import numpy as np
import torch


def per_word(d, model, v_train=()):
    rows = []
    for cond, wkey, ikey in (("seen", "word_seen", "w_seen"), ("unseen", "word_unseen", "w_unseen")):
        words = np.array(d[wkey])
        target = d[ikey].numpy()
        clip_hit = d["preds"]["clip"][cond].numpy() == target
        dp_hit = d["preds"][model][cond].numpy() == target
        dp_ok = d["preds"][model][cond].numpy() == d["labels"].numpy()
        for w in np.unique(words):
            m = words == w
            group = "imagenet100" if cond == "seen" else ("everyday_train" if str(w) in v_train else "everyday_heldout")
            rows.append({"word": str(w), "seen": int(cond == "seen"), "group": group, "n": int(m.sum()),
                         "clip_hit": 100 * clip_hit[m].mean(), "dp_hit": 100 * dp_hit[m].mean(),
                         "dp_acc": 100 * dp_ok[m].mean()})
    return rows


def wls(rows):
    X = np.array([[1.0, r["clip_hit"], r["seen"]] for r in rows])
    y = np.array([r["dp_hit"] for r in rows])
    w = np.sqrt(np.array([r["n"] for r in rows], dtype=float))
    beta, *_ = np.linalg.lstsq(X * w[:, None], y * w, rcond=None)
    return beta


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--preds", nargs="+", required=True)
    ap.add_argument("--model", default="authors")
    ap.add_argument("--out", required=True)
    ap.add_argument("--n_boot", type=int, default=2000)
    ap.add_argument("--vocab", default=None, help="vocab.json / v_train.txt: tag unseen words used in option-B training")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    v_train = set()
    if args.vocab:
        v_train = set(json.load(open(args.vocab))["v_train"]) if args.vocab.endswith(".json") else \
            {l.strip() for l in open(args.vocab) if l.strip()}
    rng = np.random.default_rng(0)
    summary = []
    for path in args.preds:
        d = torch.load(path, map_location="cpu", weights_only=False)
        name = os.path.basename(path)[len("preds_"):-3]
        rows = per_word(d, args.model, v_train)
        beta = wls(rows)
        seen_rows = [r for r in rows if r["seen"]]
        unseen_rows = [r for r in rows if not r["seen"]]
        boots = []
        for _ in range(args.n_boot):
            sample = [seen_rows[i] for i in rng.integers(len(seen_rows), size=len(seen_rows))] + \
                     [unseen_rows[i] for i in rng.integers(len(unseen_rows), size=len(unseen_rows))]
            boots.append(wls(sample)[2])
        lo, hi = np.percentile(boots, [2.5, 97.5])
        s = {"dataset": name, "model": args.model, "n_words_seen": len(seen_rows), "n_words_unseen": len(unseen_rows),
             "mean_clip_hit_seen": round(float(np.mean([r["clip_hit"] for r in seen_rows])), 2),
             "mean_clip_hit_unseen": round(float(np.mean([r["clip_hit"] for r in unseen_rows])), 2),
             "slope_on_clip_hit": round(float(beta[1]), 3),
             "seen_effect_c": round(float(beta[2]), 2), "seen_effect_ci95": [round(float(lo), 2), round(float(hi), 2)]}
        for g in ("imagenet100", "everyday_train", "everyday_heldout"):
            gr = [r for r in rows if r["group"] == g]
            if gr:
                n = sum(r["n"] for r in gr)
                s[f"{g}_words"] = len(gr)
                s[f"{g}_clip_hit"] = round(sum(r["clip_hit"] * r["n"] for r in gr) / n, 2)
                s[f"{g}_model_hit"] = round(sum(r["dp_hit"] * r["n"] for r in gr) / n, 2)
        summary.append(s)
        print(json.dumps(s))
        with open(os.path.join(args.out, f"words_{name}_{args.model}.csv"), "w", newline="") as f:
            wr = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            wr.writeheader()
            wr.writerows([{k: (round(v, 2) if isinstance(v, float) else v) for k, v in r.items()} for r in rows])
        try:
            import matplotlib
            matplotlib.use("Agg")
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(5.5, 4.5))
            for flag, label, marker in ((1, "seen (ImageNet-100 name)", "o"), (0, "unseen (length-matched)", "^")):
                pts = [r for r in rows if r["seen"] == flag]
                ax.scatter([r["clip_hit"] for r in pts], [r["dp_hit"] for r in pts], s=[6 + r["n"] / 2 for r in pts],
                           marker=marker, alpha=0.6, label=label)
            xs = np.linspace(0, 100, 50)
            for flag, style in ((1, "-"), (0, "--")):
                ax.plot(xs, beta[0] + beta[1] * xs + beta[2] * flag, style, linewidth=1)
            ax.set_xlabel("plain CLIP: % predicted as the written word")
            ax.set_ylabel(f"{args.model}: % still predicted as the written word")
            ax.set_title(f"{name}: per-word attack success")
            ax.legend(fontsize=8)
            fig.tight_layout()
            fig.savefig(os.path.join(args.out, f"words_{name}_{args.model}.png"), dpi=150)
        except ImportError:
            pass
    with open(os.path.join(args.out, f"summary_{args.model}.json"), "w") as f:
        json.dump(summary, f, indent=1)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile summarize_runs.py
"""Aggregate eval_cached.py CSVs over seeds: mean and sample std per dataset/condition, next to the paper.

Usage:
  python summarize_runs.py --results RESULTS_DIR --group repro=dp_seed0,dp_seed1,dp_seed2 --group authors=authors_token
Each name in a group is a tag written by eval_cached.py (RESULTS_DIR/<tag>.csv).
"""
import argparse
import csv
import os
import statistics as st


def read(path):
    with open(path) as f:
        return list(csv.DictReader(f))


def fmt(vals):
    m = st.mean(vals)
    return f"{m:.2f} ± {st.stdev(vals):.2f}" if len(vals) > 1 else f"{m:.2f}"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", required=True)
    ap.add_argument("--group", action="append", required=True, help="name=tag1,tag2,...")
    ap.add_argument("--out", default=None)
    args = ap.parse_args()

    groups = {}
    for g in args.group:
        name, tags = g.split("=", 1)
        groups[name] = [read(os.path.join(args.results, f"{t}.csv")) for t in tags.split(",")]

    first = next(iter(groups.values()))[0]
    keys = [(r["dataset"], r["condition"]) for r in first]
    header = ["dataset", "condition", "paper_clip", "paper_dp", "ours_clip"] + [f"{g}_dp" for g in groups]
    lines = []
    for ds, cond in keys:
        row = {"dataset": ds, "condition": cond}
        ref = next(r for r in first if r["dataset"] == ds and r["condition"] == cond)
        row["paper_clip"], row["paper_dp"], row["ours_clip"] = ref["paper_clip"], ref["paper_dp"], ref["acc_clip"]
        for g, runs in groups.items():
            vals = [float(next(r for r in run if r["dataset"] == ds and r["condition"] == cond)["acc_dp"]) for run in runs]
            row[f"{g}_dp"] = fmt(vals)
        lines.append(row)
    # condition means: mean over datasets within each run, then mean ± std over runs
    for cond in ["clean", "synthetic", "real"]:
        sub = [k for k in keys if k[1] == cond]
        if not sub:
            continue
        row = {"dataset": f"MEAN ({len(sub)} sets)", "condition": cond, "paper_clip": "", "paper_dp": ""}
        row["ours_clip"] = f"{st.mean(float(next(r for r in first if (r['dataset'], r['condition']) == k)['acc_clip']) for k in sub):.2f}"
        for g, runs in groups.items():
            per_run = [st.mean(float(next(r for r in run if (r["dataset"], r["condition"]) == k)["acc_dp"]) for k in sub)
                       for run in runs]
            row[f"{g}_dp"] = fmt(per_run)
        lines.append(row)

    width = [max(len(h), *(len(str(l.get(h, ""))) for l in lines)) for h in header]
    print("  ".join(h.ljust(w) for h, w in zip(header, width)))
    for l in lines:
        print("  ".join(str(l.get(h, "")).ljust(w) for h, w in zip(header, width)))
    if args.out:
        with open(args.out, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=header)
            w.writeheader()
            w.writerows(lines)
        print("saved", args.out)


if __name__ == "__main__":
    main()

In [ ]:
# B) Build the two training files and check the premise (everyday words fool plain CLIP more on training images)
if not os.path.exists(f"{CT}/train_vocab.pt"):
    !python merge_train.py --base {CT}/train_imagenet100.pt --attacks {CT}/train_attacks_vocab.pt --out {CT} --check
print(open(f"{CT}/attack_strength_check.json").read())

In [ ]:
# C) Training runs (finished runs are skipped, so this cell can be re-run after a disconnect)
runs = [(f"vocab_seed{s}", f"--train_file {CT}/train_vocab.pt --extra_words {CT}/v_train.txt --seed {s}") for s in (0, 1, 2)]
runs += [("ctrl_renderer", f"--train_file {CT}/train_ctrl.pt --seed 0"),
         ("ctrl_distractors", f"--train_file {CT}/train_ctrl.pt --extra_words {CT}/v_train.txt --seed 0")]
for tag, a in runs:
    if os.path.exists(f"{W}/runs/{tag}/dp_token.pt"):
        print(tag, "already trained"); continue
    !python train_dp_cached.py {a} --out {W}/runs/{tag} 2>&1 | tail -2
for r in sorted(glob.glob(f"{W}/runs/*/train_log.json")):
    m = json.load(open(r)); print(f'{r.split("/")[-2]:<22} train {m["train_seconds"]:>7}s | peak GPU {m["peak_gpu_mem_mb"]} MB | candidates {m["n_candidates"]}')

In [ ]:
# C2) Five untrained random vectors (no training needed): how much accuracy does ANY extra token buy?
# Notebook 05 showed that one random N(0, 0.02) vector already lifts the real-world mean from 46.3 to 53.1.
for s in range(5):
    path = f"{W}/runs/ctrl_random_s{s}/dp_token.pt"
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        g = torch.Generator().manual_seed(1000 + s)
        torch.save((torch.randn(1, 512, generator=g) * 0.02).half(), path)
    if not os.path.exists(f"{W}/results/ctrl_random_s{s}.csv"):
        !python eval_cached.py --cache {CACHE} --token {path} --tag ctrl_random_s{s} --out {W}/results > {W}/results/eval_ctrl_random_s{s}.txt
!python summarize_runs.py --results {W}/results --group authors=authors_token --group repro=dp_seed0,dp_seed1,dp_seed2 --group identity_only=ctrl_identity_only --group random5=ctrl_random_s0,ctrl_random_s1,ctrl_random_s2,ctrl_random_s3,ctrl_random_s4 --out {W}/results/summary_controls.csv

In [ ]:
# D1) All cached test sets, next to the reproduction (notebook 05) and the authors' vector
for tag in ["vocab_seed0", "vocab_seed1", "vocab_seed2", "ctrl_renderer", "ctrl_distractors"]:
    if not os.path.exists(f"{W}/results/{tag}.csv"):
        !python eval_cached.py --cache {CACHE} --token {W}/runs/{tag}/dp_token.pt --tag {tag} --out {W}/results > {W}/results/eval_{tag}.txt
!python summarize_runs.py --results {W}/results --group authors=authors_token --group repro=dp_seed0,dp_seed1,dp_seed2 --group vocab=vocab_seed0,vocab_seed1,vocab_seed2 --group random5=ctrl_random_s0,ctrl_random_s1,ctrl_random_s2,ctrl_random_s3,ctrl_random_s4 --group ctrl_renderer=ctrl_renderer --group ctrl_distractors=ctrl_distractors --out {W}/results/summary_06.csv

In [ ]:
# D2) Seen / unseen sets of notebook 04 (unseen words split into everyday-train vs held-out in the word analysis)
SU = " ".join(sorted(glob.glob(f"{W}/cache_su/su_*.pt")))
toks = "authors=learned_token/dp_vit-b32.pt " + " ".join(f"{t}={W}/runs/{t}/dp_token.pt" for t in
        ["dp_seed0", "dp_seed1", "dp_seed2", "vocab_seed0", "vocab_seed1", "vocab_seed2", "ctrl_renderer", "ctrl_distractors"])
!python eval_seen_unseen.py --data {SU} --tokens {toks} --out {W}/results/seen_unseen_06
PREDS = " ".join(sorted(glob.glob(f"{W}/results/seen_unseen_06/preds_*.pt")))
for tag in ["authors", "dp_seed0", "vocab_seed0", "vocab_seed1", "vocab_seed2", "ctrl_distractors"]:
    !python analyze_words.py --preds {PREDS} --model {tag} --vocab {CT}/vocab.json --out {W}/results/words_06 2>&1 | tail -2

In [ ]:
# D3) Pack results
!cd {W} && zip -q -r results_06.zip results/summary_06.csv results/*.csv results/seen_unseen_06 results/words_06 runs/*/train_log.json && ls -la {W}/results_06.zip